# YAMNet 소리 분류 체험 노트북 — 하드웨어 없이 랩탑으로 배우기

이 노트북은 **HearSafe** (청각장애인 위험 소리 방향 감지 웨어러블) 프로젝트에서
소리를 "무엇인지" 판별하는 부분(YAMNet 소리 분류)을, 안경형 센서 하드웨어
(ESP32-S3 + 마이크 4개) 없이 **본인 노트북만으로** 체험하고 이해하기 위한
자료입니다.

실제 제품에서는 안경의 마이크 1개가 잡은 소리를 노트북(`dashboard/serve.py`)이
받아 YAMNet으로 분류합니다. 이 노트북에서는 그 "노트북이 하는 일"만 따로
떼어내서, 여러분의 랩탑 마이크나 준비된 테스트 사운드로 똑같이 해봅니다.

## 이 노트북에서 다루는 순서
1. **소리 준비** — 테스트 사운드를 불러오거나 직접 녹음
2. **YAMNet으로 분류하기** — 무슨 소리인지 판별
3. **근거 데이터** — 왜 그 소리로 판별됐는지 (점수 분포, 시간에 따른 변화, 커스텀 소리 유사도)
4. **내부적 처리 방법** — YAMNet이 파형을 어떻게 숫자로 바꾸는지 (멜 스펙트로그램 직접 시각화)
5. **개선을 위해 고민해야 할 점** — 실제 프로젝트가 겪고 있는 튜닝 이슈와 실습 과제

배경 원리(TDOA/GCC-PHAT 방향 계산 등)는 이 노트북 범위가 아닙니다 —
`docs/factsheet.md`를 참고하세요. 여기서는 오직 **소리 분류(YAMNet)** 만 다룹니다.


## 0. 실행 준비

터미널에서 아래 명령으로 필요한 패키지를 설치하세요:

```powershell
python -m pip install tensorflow tensorflow-hub numpy matplotlib jupyter sounddevice
```

- `tensorflow`, `tensorflow-hub`: YAMNet 모델 실행 (최초 실행 시 모델을 인터넷에서
  내려받아 캐시합니다 — 몇 분 걸릴 수 있습니다)
- `matplotlib`: 이 노트북의 그래프
- `sounddevice`: (선택) 랩탑 마이크로 직접 녹음하고 싶을 때만 필요 — 없어도
  노트북이 즉석에서 만들어내는 테스트 사운드로 전체 실습이 가능합니다

이 노트북은 **어디에 두고 열어도** 동작합니다 — Drive 등에서 이 파일 하나만
받았다면 "독립 실행 모드"로 자동 전환되어 필요한 값을 노트북 안에 내장된
사본으로 사용합니다. HearSafe 저장소의 `dashboard` 폴더 안에서 열면
`classifier.py`, `custom_sounds.json`을 자동으로 찾아 항상 최신 값을 씁니다
("저장소 모드") — 어느 쪽이든 셀 내용과 실습 순서는 동일합니다.

셀은 위에서부터 순서대로 실행하세요 (`Shift+Enter`).


In [ ]:
import json
import os
import wave

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

try:
    import tensorflow as tf
    import tensorflow_hub as hub
except ImportError as e:
    raise ImportError(
        "tensorflow / tensorflow-hub가 설치되어 있지 않습니다. 아래 명령으로 설치 후 "
        "커널을 다시 시작하세요:\n"
        "  python -m pip install tensorflow tensorflow-hub numpy matplotlib jupyter sounddevice"
    ) from e


In [ ]:
# classifier.py가 같은 폴더에 있으면(저장소의 dashboard 폴더에서 실행) 최신 값을 그대로
# 가져오고, 없으면(독립 실행 모드) 아래 내장 스냅샷을 사용합니다.
try:
    from classifier import KO, DANGER, MIN_SCORE, WINDOW_SEC, HOP_SEC
    print("classifier.py에서 설정을 불러왔습니다 (저장소 모드)")
except ImportError:
    KO = {'Fire alarm': '🔥 화재경보', 'Smoke detector, smoke alarm': '🔥 화재경보', 'Siren': '🚨 사이렌', 'Civil defense siren': '🚨 민방위 사이렌', 'Ambulance (siren)': '🚑 구급차 사이렌', 'Police car (siren)': '🚓 경찰차 사이렌', 'Fire engine, fire truck (siren)': '🚒 소방차 사이렌', 'Vehicle horn, car horn, honking': '📯 자동차 경적', 'Air horn, truck horn': '📯 트럭 경적', 'Doorbell': '🔔 초인종', 'Ding-dong': '🔔 초인종', 'Knock': '🚪 노크', 'Clapping': '👏 박수', 'Applause': '👏 박수', 'Hands': '👏 손뼉', 'Speech': '💬 말소리', 'Shout': '📢 외침', 'Yell': '📢 외침', 'Screaming': '😱 비명', 'Baby cry, infant cry': '👶 아기 울음', 'Dog': '🐕 개', 'Bark': '🐕 개 짖음', 'Glass': '🍷 유리', 'Shatter': '💥 깨지는 소리', 'Alarm': '⏰ 경보음', 'Alarm clock': '⏰ 알람시계', 'Beep, bleep': '🔊 삐 소리', 'Buzzer': '🔊 버저', 'Boom': '💥 쿵', 'Whistling': '😗 휘파람', 'Whistle': '😗 휘파람', 'Silence': '조용함', 'Music': '🎵 음악', 'Water': '💧 물소리', 'Typing': '⌨️ 타이핑'}
    DANGER = {'🚒 소방차 사이렌', '🚓 경찰차 사이렌', '🐕 개', '😗 휘파람', '😱 비명', '💥 깨지는 소리', '🚨 민방위 사이렌', '🔥 화재경보', '💥 쿵', '🚨 사이렌', '🚑 구급차 사이렌', '⏰ 경보음', '🐕 개 짖음', '📯 자동차 경적', '📯 트럭 경적'}
    MIN_SCORE = 0.1
    WINDOW_SEC = 0.975
    HOP_SEC = 0.5
    print("독립 실행 모드입니다 (노트북에 내장된 classifier.py 스냅샷 사용)")

print(f"MIN_SCORE(최소 보고 문턱)={MIN_SCORE}, WINDOW_SEC={WINDOW_SEC}s, HOP_SEC={HOP_SEC}s (serve.py 실시간 스트리밍 기준)")


In [ ]:
# 색상: dataviz 스타일 가이드의 검증된 팔레트 값을 그대로 사용
CAT = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
STATUS_CRITICAL = "#d03b3b"
INK = "#0b0b0b"
INK_SECONDARY = "#52514e"
MUTED = "#898781"
GRID = "#e1e0d9"
SURFACE = "#fcfcfb"
SEQ_BLUE = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
MEL_CMAP = LinearSegmentedColormap.from_list("seq_blue", SEQ_BLUE)

# matplotlib 기본 폰트(DejaVu Sans)에는 한글 글리프가 없어 그래프 라벨이 네모로 깨집니다.
# OS별로 기본 설치된 한글 폰트를 찾아서 지정합니다.
import matplotlib.font_manager as fm

_KOREAN_FONT_CANDIDATES = [
    "Malgun Gothic",       # Windows 기본
    "AppleGothic",         # macOS 기본
    "NanumGothic", "Noto Sans CJK KR", "Noto Sans KR",  # Linux (설치되어 있는 경우)
]
_available_fonts = {f.name for f in fm.fontManager.ttflist}
_kr_font = next((f for f in _KOREAN_FONT_CANDIDATES if f in _available_fonts), None)
if _kr_font:
    plt.rcParams["font.family"] = _kr_font
else:
    print(
        "[경고] 한글 폰트를 찾지 못했습니다 — 그래프의 한글 라벨이 네모(□)로 보일 수 있습니다.\n"
        "Windows는 보통 'Malgun Gothic'이 기본 설치되어 있는데, matplotlib 폰트 캐시가 오래되면 "
        "못 찾을 수 있습니다. 아래 폴더를 지운 뒤 커널을 재시작해보세요:\n"
        f"  {matplotlib.get_cachedir()}\n"
        "그래도 안 되면 나눔고딕 등 한글 폰트를 설치한 뒤 다시 시도하세요."
    )
plt.rcParams["axes.unicode_minus"] = False  # 한글 폰트는 유니코드 마이너스(−) 글리프가 없는 경우가 많음

plt.rcParams.update({
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "axes.edgecolor": GRID,
    "axes.labelcolor": INK_SECONDARY,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "text.color": INK,
    "axes.grid": True,
    "grid.color": GRID,
    "grid.linewidth": 0.8,
    "font.size": 10,
})


def _strip_spines(ax):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)


def plot_waveform(x, sr, title=""):
    t = np.arange(len(x)) / sr
    fig, ax = plt.subplots(figsize=(9, 2.2))
    ax.plot(t, x, color=CAT[0], linewidth=1)
    ax.set_xlim(0, t[-1] if len(t) else 1)
    ax.set_ylim(-1.05, 1.05)
    ax.set_xlabel("시간 (초)")
    ax.set_ylabel("진폭")
    ax.set_title(title, loc="left", fontsize=11, color=INK)
    _strip_spines(ax)
    plt.tight_layout()
    plt.show()


def plot_top_scores(items, title="", threshold=None):
    """items: [(labelKo, score, is_danger), ...] 점수 내림차순"""
    items = items[::-1]  # barh는 아래부터 그려지므로 뒤집어서 1등이 위로
    labels = [it[0] for it in items]
    scores = [it[1] for it in items]
    colors = [STATUS_CRITICAL if it[2] else CAT[0] for it in items]
    fig, ax = plt.subplots(figsize=(7, 0.42 * len(items) + 1))
    bars = ax.barh(labels, scores, color=colors, height=0.6)
    for b, s in zip(bars, scores):
        ax.text(s + 0.015, b.get_y() + b.get_height() / 2, f"{s:.2f}",
                 va="center", fontsize=9, color=INK_SECONDARY)
    if threshold is not None:
        ax.axvline(threshold, color=MUTED, linewidth=1, linestyle="--")
        ax.text(threshold, len(items) - 0.4, f" 최소 보고 문턱 {threshold}",
                 fontsize=8, color=MUTED)
    ax.set_xlim(0, 1.05)
    ax.set_xlabel("YAMNet 점수 (0~1, 클래스별 독립 시그모이드 — 합이 1이 아님)")
    ax.set_title(title, loc="left", fontsize=11, color=INK)
    _strip_spines(ax)
    plt.tight_layout()
    plt.show()


def plot_score_timeline(times, series, title=""):
    """series: [(labelKo, [score, ...]), ...]"""
    fig, ax = plt.subplots(figsize=(9, 3))
    for i, (label, ys) in enumerate(series):
        ax.plot(times, ys, color=CAT[i % len(CAT)], linewidth=2, label=label, marker="o", markersize=3)
    ax.set_xlabel("시간 (초)")
    ax.set_ylabel("YAMNet 점수")
    ax.set_ylim(0, 1.02)
    ax.set_title(title, loc="left", fontsize=11, color=INK)
    ax.legend(frameon=False, loc="upper right", fontsize=9)
    _strip_spines(ax)
    plt.tight_layout()
    plt.show()


def plot_internal_pipeline(x, sr, log_mel, title=""):
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(9, 5), sharex=True, gridspec_kw={"height_ratios": [1, 2]})
    t = np.arange(len(x)) / sr
    ax1.plot(t, x, color=CAT[0], linewidth=1)
    ax1.set_ylabel("진폭")
    ax1.set_title(title, loc="left", fontsize=11, color=INK)
    _strip_spines(ax1)

    n_frames = log_mel.shape[0]
    frame_hop_sec = 0.010  # YAMNet 내부 STFT hop (25ms 창 / 10ms 이동)
    t_end = max(n_frames * frame_hop_sec, frame_hop_sec)
    im = ax2.imshow(log_mel.T, origin="lower", aspect="auto",
                     extent=[0, t_end, 0, log_mel.shape[1]], cmap=MEL_CMAP)
    ax2.set_xlabel("시간 (초)")
    ax2.set_ylabel("멜 밴드 (0=저음 ~ 63=고음, 125~7500Hz)")
    cbar = fig.colorbar(im, ax=ax2, fraction=0.035, pad=0.02)
    cbar.set_label("log-mel 에너지", color=INK_SECONDARY, fontsize=8)
    plt.tight_layout()
    plt.show()

print("그래프 유틸 준비 완료")


## 1. 소리 준비

세 가지 방법 중 하나로 분석할 소리를 준비합니다.

- **A. 합성 테스트 사운드** — 노트북이 그 자리에서 만들어내는 박수/경적/화재경보
  (HearSafe 실측에 쓴 `tools/make_test_sounds.py`와 같은 방식 — 파일 다운로드가
  필요 없어 어떤 환경에서도 바로 됩니다)
- **B. 랩탑 마이크로 직접 녹음** — `sounddevice`가 설치되어 있으면 바로 녹음 가능
- **C. 본인 wav 파일** — 16bit PCM mono wav 파일 경로를 직접 지정


In [ ]:
def resample_linear(x, sr_from, sr_to):
    """의존성을 최소화하기 위한 간단 선형보간 리샘플. 프로덕션 품질은 아니지만
    이 노트북의 시연 목적에는 충분합니다 (실제로는 polyphase 필터 등을 씁니다)."""
    if sr_from == sr_to:
        return x.astype(np.float32)
    duration = len(x) / sr_from
    n_new = max(1, int(round(duration * sr_to)))
    t_old = np.linspace(0, duration, num=len(x), endpoint=False)
    t_new = np.linspace(0, duration, num=n_new, endpoint=False)
    return np.interp(t_new, t_old, x).astype(np.float32)


def load_wav_mono16k(path):
    """16bit PCM wav → float32 [-1,1] mono 16kHz"""
    with wave.open(path, "rb") as w:
        sr = w.getframerate()
        n = w.getnframes()
        sampwidth = w.getsampwidth()
        nch = w.getnchannels()
        raw = w.readframes(n)
    if sampwidth != 2:
        raise ValueError(f"16bit PCM wav만 지원합니다 (현재 {sampwidth * 8}bit): {path}")
    pcm = np.frombuffer(raw, dtype="<i2")
    if nch > 1:
        pcm = pcm.reshape(-1, nch).mean(axis=1)
    x = pcm.astype(np.float32) / 32768.0
    return resample_linear(x, sr, 16000)


print("wav 로더 준비 완료 — 본인 파일이 있다면: waveform = load_wav_mono16k(r'경로')")


In [ ]:
# tools/make_test_sounds.py와 같은 방식으로 합성 사운드를 직접 만듭니다 — 파일이
# 전혀 필요 없어서, 이 노트북 파일 하나만 있어도 바로 실습할 수 있습니다.
SYNTH_SR = 16000


def _tone(freq, sec, amp=0.8, sr=SYNTH_SR):
    t = np.arange(int(sr * sec)) / sr
    return (amp * np.sin(2 * np.pi * freq * t)).astype(np.float32)


def _silence(sec, sr=SYNTH_SR):
    return np.zeros(int(sr * sec), dtype=np.float32)


def make_fire_t3():
    """화재경보 ISO 8201 T-3 패턴: 3.2kHz를 0.5초 on/off로 3번 울리고 1.5초 쉬기, 2회 반복."""
    one_cycle = [_tone(3200, 0.5), _silence(0.5)] * 3 + [_silence(1.0)]
    return np.concatenate(one_cycle * 2)


def make_horn():
    """자동차 경적 근사: 400Hz 기본파 + 배음(800/1200Hz) 1.5초."""
    t = np.arange(int(SYNTH_SR * 1.5)) / SYNTH_SR
    v = (0.5 * np.sin(2 * np.pi * 400 * t)
         + 0.3 * np.sin(2 * np.pi * 800 * t)
         + 0.2 * np.sin(2 * np.pi * 1200 * t))
    return (v * 0.9).astype(np.float32)


def make_clap():
    """박수 근사: 백색잡음 버스트(30ms) 3번, 0.5초 간격."""
    rng = np.random.RandomState(42)
    n = int(SYNTH_SR * 0.03)
    envelope = np.exp(-6.0 * np.arange(n) / n)
    burst = (rng.uniform(-1, 1, n) * envelope).astype(np.float32)
    return np.concatenate([burst, _silence(0.5)] * 3)


SYNTH_SOUNDS = {
    "fire_t3 (화재경보 T-3, 합성)": make_fire_t3,
    "horn (경적, 합성)": make_horn,
    "clap (박수, 합성)": make_clap,
}
print("합성 가능한 테스트 사운드:", ", ".join(SYNTH_SOUNDS))


In [ ]:
# ▼▼ 여기를 바꿔서 다른 소리로 실습해보세요 ▼▼
SOUND_NAME = "fire_t3 (화재경보 T-3, 합성)"

waveform = SYNTH_SOUNDS[SOUND_NAME]()
sr = SYNTH_SR
print(f"'{SOUND_NAME}' 생성 완료 — {len(waveform)/sr:.2f}초, {len(waveform)} 샘플")
plot_waveform(waveform, sr, title=f"파형 — {SOUND_NAME}")


### (선택) B. 랩탑 마이크로 직접 녹음하기

`sounddevice`가 설치되어 있으면 실행하세요. 마이크 권한 요청이 뜰 수 있습니다.
녹음이 끝나면 `waveform` 변수가 방금 녹음한 소리로 교체됩니다 — 이후 셀들은
그대로 다시 실행하면 됩니다.


In [ ]:
RECORD_SECONDS = 3

try:
    import sounddevice as sd

    print(f"{RECORD_SECONDS}초간 녹음합니다 — 마이크에 대고 소리를 내보세요 (박수, 목소리 등)")
    rec = sd.rec(int(RECORD_SECONDS * sr), samplerate=sr, channels=1, dtype="float32")
    sd.wait()
    waveform = rec[:, 0]
    print("녹음 완료")
    plot_waveform(waveform, sr, title="파형 — 방금 녹음한 소리")
except ImportError:
    print("sounddevice가 설치되어 있지 않습니다 — 이 셀은 건너뛰고 위의 테스트 사운드를 사용하세요.")
    print("설치하려면: python -m pip install sounddevice")
except Exception as e:
    print(f"녹음 실패 (마이크 연결/권한을 확인하세요): {e}")


## 2. YAMNet으로 분류하기

`dashboard/serve.py`가 실시간으로 하는 일을 그대로 재현합니다: 파형을 YAMNet에
넣으면 **521개 AudioSet 클래스에 대한 점수(scores)**, **1024차원 임베딩
(embeddings)**, 그리고 **내부에서 쓴 log-mel 스펙트로그램**까지 3가지를
한 번에 돌려받습니다. 세 번째 출력은 `classifier.py`에서는 쓰지 않고 버리는데
(`_`로 받음), 이 노트북의 4번 섹션에서 "내부적으로 무슨 일이 일어나는가"를
보여줄 때 바로 이 값을 사용합니다.


In [ ]:
print("YAMNet 모델을 불러오는 중… (최초 1회는 인터넷에서 내려받아 캐시하므로 다소 걸립니다)")
yamnet_model = hub.load("https://tfhub.dev/google/yamnet/1")
class_map_path = yamnet_model.class_map_path().numpy().decode("utf-8")
import csv
with tf.io.gfile.GFile(class_map_path) as f:
    class_names = [row["display_name"] for row in csv.DictReader(f)]
print(f"준비 완료 — AudioSet {len(class_names)}개 클래스")


In [ ]:
def classify_clip(x):
    """x: float32 mono 16kHz 파형, 길이 제한 없음.
    YAMNet이 내부적으로 0.96초 창 / 0.48초 간격 패치로 알아서 나눠 처리하므로,
    실시간 스트리밍(classifier.py)과 달리 우리가 직접 슬라이딩 윈도우를 관리할
    필요가 없습니다 — 클립 전체를 한 번에 넣으면 됩니다."""
    scores, embeddings, log_mel = yamnet_model(x)
    return scores.numpy(), embeddings.numpy(), log_mel.numpy()


def top_scores(mean_scores, n=5):
    idx = mean_scores.argsort()[-n:][::-1]
    out = []
    for i in idx:
        name = class_names[i]
        ko = KO.get(name, name)
        out.append((ko, float(mean_scores[i]), ko in DANGER))
    return out


scores, embeddings, log_mel = classify_clip(waveform)
print(f"scores.shape={scores.shape}  (패치 수 x 521클래스 — 이 클립이 내부적으로 {scores.shape[0]}개 패치로 나뉘었다는 뜻)")
print(f"embeddings.shape={embeddings.shape}  (패치 수 x 1024차원)")
print(f"log_mel.shape={log_mel.shape}  (프레임 수 x 64 멜밴드)")

mean_scores = scores.mean(axis=0)  # serve.py와 동일: 패치별 점수를 평균내서 최종 판정
top5 = top_scores(mean_scores, n=5)
print()
print(f"'{SOUND_NAME}' 분류 결과 (상위 5개):")
for label, score, danger in top5:
    flag = " [위험 신호로 분류됨]" if danger else ""
    print(f"  {score:.3f}  {label}{flag}")

plot_top_scores(top5, title=f"YAMNet 분류 결과 — {SOUND_NAME}", threshold=MIN_SCORE)


## 3. 근거 데이터 — 왜 이 소리로 판별됐을까?

YAMNet의 점수는 **다중 라벨(multi-label) 시그모이드**입니다 — 이미지 분류처럼
"1등이 51%, 나머지 49%를 나눠 가짐" 방식(softmax)이 아니라, **클래스마다
독립적으로 0~1 점수**를 매깁니다. 그래서 한 소리에 여러 클래스가 동시에
0.8, 0.7처럼 높게 나올 수 있고, 반대로 top1이 0.15 정도로 낮으면
"이 소리가 뭔지 애매하다"는 뜻입니다. `classifier.py`가 `MIN_SCORE=0.10`
미만은 아예 보고하지 않는 이유가 여기 있습니다 — 애매한 판정을 사용자에게
그대로 노출하지 않기 위해서입니다.


In [ ]:
top15 = top_scores(mean_scores, n=15)
plot_top_scores(top15, title=f"전체 상위 15개 클래스 점수 — {SOUND_NAME}", threshold=MIN_SCORE)


### 시간에 따른 점수 변화

`scores`는 패치(0.48초 간격)마다 하나씩 있습니다. 상위 클래스의 점수가
시간에 따라 어떻게 움직이는지 보면 "왜 그 순간에 그렇게 판정했는지"가
더 명확히 보입니다. 예를 들어 화재경보(T-3 패턴)는 삐-삐-삐 울리고 쉬기를
반복하므로 점수가 주기적으로 오르내려야 정상입니다 — 만약 평평하다면
합성음이 실제 패턴을 제대로 재현하지 못했다는 뜻일 수 있습니다.


In [ ]:
PATCH_HOP_SEC = 0.48  # YAMNet 내부 패치 간격 (고정값)
n_patches = scores.shape[0]
patch_times = np.arange(n_patches) * PATCH_HOP_SEC + PATCH_HOP_SEC

# 클립 전체 평균 기준 상위 3개 클래스를 골라 그 시간별 궤적을 그린다
top3_idx = mean_scores.argsort()[-3:][::-1]
series = []
for i in top3_idx:
    label = KO.get(class_names[i], class_names[i])
    series.append((label, scores[:, i].tolist()))

if n_patches >= 2:
    plot_score_timeline(patch_times, series, title=f"시간에 따른 상위 클래스 점수 — {SOUND_NAME}")
else:
    print(f"이 클립은 {n_patches}개 패치뿐이라 시간별 그래프 대신 더 긴 소리(예: fire_t3)로 시도해보세요.")


### 커스텀 소리(few-shot) 판정 근거

대회 시연에서 쓴 "나만의 소리 등록"(예: 쿠쿠 밥솥 완료음)은 YAMNet 521클래스에
없는 소리라서 재학습 없이 **임베딩 코사인 유사도**로 판별합니다. 등록할 때
저장한 지문(임베딩)과 지금 들어온 소리의 임베딩이 얼마나 비슷한지가 근거입니다.


In [ ]:
if os.path.exists("custom_sounds.json"):
    with open("custom_sounds.json", encoding="utf-8") as f:
        custom_sounds = json.load(f)
    print("custom_sounds.json을 불러왔습니다 (저장소 모드 — 실제 등록된 최신 소리)")
else:
    # 독립 실행 모드 스냅샷 — HearSafe 대시보드에 실제 등록된 쿠쿠 밥솥 완료음 지문
    custom_sounds = json.loads('[{"name": "쿠쿠", "labelKo": "📌 쿠쿠", "danger": true, "threshold": 0.72, "embeddings": [[0.0, 0.007278891280293465, 0.04095999151468277, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.00014970633492339402, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.00019349400827195495, 0.0027638832107186317, 0.0017554264049977064, 0.0, 0.0, 0.005198573227971792, 0.0, 0.0006266598938964307, 0.0, 0.00016191987378988415, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.00015004001033958048, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0012690621661022305, 0.0, 0.0, 0.0, 0.0, 0.0008467069710604846, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.09556006640195847, 0.0, 0.0, 0.00039173176628537476, 0.0, 0.002173504326492548, 0.0, 0.19678980112075806, 0.02339334227144718, 0.0, 0.0, 0.0, 0.005371213890612125, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0009507980430498719, 0.0, 0.06561512500047684, 0.0, 0.0, 0.0, 0.017560862004756927, 0.0, 0.005460289306938648, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.271939754486084, 0.18076126277446747, 0.08602584153413773, 0.06980586051940918, 0.0, 0.0, 0.0, 0.02238120697438717, 0.07475161552429199, 0.0, 0.13113532960414886, 0.0, 0.0, 0.08036835491657257, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.005401817616075277, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0012992037227377295, 0.0, 0.011688838712871075, 0.0, 0.0, 0.0313672311604023, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.027502315118908882, 0.0, 0.004143405705690384, 0.0, 0.0, 0.037984155118465424, 0.0069364565424621105, 0.0, 0.001477140816859901, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.016422362998127937, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.05681486055254936, 0.24885323643684387, 0.0, 0.02846616506576538, 0.0, 0.0, 0.0, 0.06750395148992538, 0.0, 0.036049019545316696, 0.0014370566932484508, 0.0005031253094784915, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.02074105478823185, 0.0006102636107243598, 0.0, 0.012365516275167465, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.03051694482564926, 0.0, 0.0, 0.0, 0.0, 0.0067161633633077145, 0.0, 0.0001315588306169957, 0.02363225258886814, 0.0, 0.001991212833672762, 0.0, 0.0011606690241023898, 0.0, 0.0, 0.0, 0.0, 0.010770989581942558, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.14136645197868347, 0.0, 0.0080479821190238, 0.002348904265090823, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0662197396159172, 0.0, 0.03086632862687111, 0.0, 0.0, 0.0, 0.0, 0.009290113113820553, 0.0014607020420953631, 0.0, 0.0, 0.21610519289970398, 0.061034731566905975, 0.0, 0.0, 0.0, 0.10504800826311111, 0.0, 0.0, 0.0048866672441363335, 0.0, 0.0027498863637447357, 0.0, 0.0, 0.0, 0.0, 0.0, 0.00921083614230156, 0.0, 0.0, 0.00804683193564415, 0.0, 0.0, 0.0, 0.0, 0.10319317132234573, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.005954318679869175, 0.0, 0.0, 0.0, 0.056085992604494095, 0.0, 0.06869383156299591, 0.0, 0.0, 0.01553609874099493, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.014982344582676888, 0.0005409311270341277, 0.0, 0.0, 0.0, 0.029959920793771744, 0.0017414639005437493, 0.010572179220616817, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.002629727590829134, 0.0, 0.0, 0.0, 0.05453015863895416, 0.042327042669057846, 0.017881808802485466, 0.0, 0.018548976629972458, 0.005700771231204271, 0.0, 0.0, 0.0, 0.0, 0.047676146030426025, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.03919697180390358, 0.0, 0.0, 0.0, 0.02428501471877098, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.15616479516029358, 0.006362913642078638, 0.0, 0.0, 0.0, 0.022065281867980957, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.025994056835770607, 0.0, 0.0026363751385360956, 0.0, 0.0, 0.0, 0.00017623625171836466, 0.0007163066766224802, 0.0, 0.18511228263378143, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.15274688601493835, 0.0, 0.0, 0.0, 0.0, 0.006018955260515213, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 8.955071825766936e-05, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.10492527484893799, 0.0, 0.0, 0.0, 0.0, 0.001200166647322476, 0.0, 0.033964600414037704, 0.0, 0.0, 0.0, 0.0, 0.0, 0.02376481704413891, 0.0024615400470793247, 0.0, 0.0, 0.0, 0.0, 0.0, 0.003637762973085046, 0.0, 0.0, 0.00137313362210989, 0.0, 0.19202879071235657, 0.00019339002028573304, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.01705753058195114, 0.0, 0.004403859376907349, 0.0, 0.0, 0.0, 0.0, 0.0, 0.004774766508489847, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0024704993702471256, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.013899658806622028, 0.0018701640656217933, 0.0, 0.0, 0.0, 0.0, 0.019555427134037018, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.007350083906203508, 0.12149124592542648, 0.0, 0.0, 0.0003279224911238998, 0.0, 0.0, 0.0025139644276350737, 0.0, 0.0, 0.0, 0.0, 0.0021474743261933327, 0.0, 0.05743486061692238, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.010486212559044361, 0.0, 0.011374135501682758, 0.0, 0.0, 0.0, 0.0, 0.019928954541683197, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.00493469275534153, 0.0, 0.0, 0.0, 0.0, 0.09035735577344894, 0.008512554690241814, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0263291634619236, 0.0, 0.0, 0.0, 0.0, 0.0, 0.07716881483793259, 0.0, 0.0, 0.0013559571234509349, 0.012213082984089851, 0.015324531123042107, 0.0, 0.0, 0.0, 0.0, 0.0001275907561648637, 0.0, 0.0, 0.0005282511119730771, 0.0, 0.01889185979962349, 0.0, 0.011169785633683205, 0.0, 0.0874975323677063, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.009095372632145882, 0.0, 0.0, 0.01819968782365322, 0.0, 0.0, 0.0037037120200693607, 0.0, 0.059834375977516174, 0.0, 0.0, 0.0, 0.01330831740051508, 0.0, 0.0, 0.0, 0.0, 0.016682567074894905, 0.029504330828785896, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.015845581889152527, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.00985756702721119, 0.07955332100391388, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.07827772200107574, 0.0, 0.045411743223667145, 0.008441333658993244, 0.0, 0.0, 0.0, 0.0014439275255426764, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.004258341621607542, 0.06016344577074051, 0.0, 0.0, 0.0, 0.009157848544418812, 0.0015079939039424062, 0.0, 0.0, 0.0, 0.0, 0.009862024337053299, 0.0, 0.0, 0.0, 0.00600847601890564, 0.0, 0.0, 0.0, 0.0, 0.009442800655961037, 0.02150702476501465, 0.0, 0.028719153255224228, 0.0, 0.0, 0.0, 0.021455949172377586, 0.0, 0.0, 0.09164615720510483, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.00821611937135458, 0.0, 0.0, 0.04422849416732788, 0.0, 0.04152875766158104, 0.18089352548122406, 0.026836836710572243, 0.0, 0.0, 0.0, 0.008969870395958424, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.03736299276351929, 0.0, 0.0, 0.0, 0.0, 0.04127310588955879, 0.0, 0.02029266208410263, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0029930933378636837, 0.0, 0.008229843340814114, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.003077947534620762, 0.0, 0.0026551508344709873, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.036631107330322266, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0027008620090782642, 0.011797625571489334, 0.0, 0.004289806354790926, 0.0, 0.0, 0.0, 0.25908124446868896, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.008330508135259151, 0.0, 0.031172655522823334, 0.0, 0.0, 0.0, 0.00464734947308898, 0.0, 0.022827038541436195, 0.0, 0.0, 0.016137249767780304, 0.005630966741591692, 0.04967886209487915, 0.0, 0.0, 0.11779052764177322, 0.0, 0.0, 0.0, 0.0, 8.199279545806348e-05, 0.0, 0.0, 0.0, 0.0, 0.0, 0.003525050124153495, 0.07011903077363968, 0.22558900713920593, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0005248291417956352, 0.009968757629394531, 0.0, 0.00010153818584512919, 0.0, 0.03925599902868271, 0.0, 0.03641905635595322, 0.07298246771097183, 0.05026482790708542, 0.0, 0.0, 0.014103014953434467, 0.0, 0.0, 0.0, 0.0, 0.0, 0.16205909848213196, 0.0, 0.0, 0.0, 0.0, 0.013458937406539917, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.08016928285360336, 0.05943060293793678, 0.0, 0.0, 0.0, 0.2229752093553543, 0.0016862500924617052, 0.0, 0.018526630476117134, 5.6825516367098317e-05, 0.0, 0.0, 0.0, 0.0, 0.0, 0.07294920831918716, 0.0, 0.0, 0.0, 0.0067297485657036304, 0.0, 0.0, 0.0, 0.0, 0.0, 0.050149425864219666, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.023421261459589005, 0.0, 0.0, 0.003499404527246952, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.10992972552776337, 0.0, 0.14248555898666382, 0.006186304613947868, 0.00878643337637186, 0.0, 0.0, 0.0, 0.011253081262111664, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.013291939161717892, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.003324915422126651, 0.0, 0.0, 0.0, 0.14283275604248047, 0.0, 0.0, 0.057562582194805145, 0.0, 0.15196850895881653, 0.12224705517292023, 0.0, 0.0, 0.0, 0.0, 0.04370906203985214, 0.0, 0.03668467700481415, 0.0, 0.0, 0.0, 0.0, 0.06399622559547424, 0.0, 0.0, 0.0]], "clip": "쿠쿠.wav"}]')
    print("독립 실행 모드 — 노트북에 내장된 예시 지문(쿠쿠 밥솥)을 사용합니다")
mean_embedding = embeddings.mean(axis=0)
mean_embedding = mean_embedding / (np.linalg.norm(mean_embedding) + 1e-9)

if not custom_sounds:
    print("등록된 커스텀 소리가 없습니다 (custom_sounds.json이 비어있음)")
else:
    print(f"'{SOUND_NAME}' vs 등록된 커스텀 소리 {len(custom_sounds)}개 — 코사인 유사도:")
    rows = []
    for c in custom_sounds:
        sims = [float(np.dot(mean_embedding, np.asarray(t, dtype=np.float32))) for t in c["embeddings"]]
        best = max(sims) if sims else 0.0
        th = c.get("threshold", 0.72)
        rows.append((c["labelKo"], best, best >= th))
        verdict = "인식됨" if best >= th else "미달"
        print(f"  {best:.3f}  (문턱 {th})  {c['labelKo']}  → {verdict}")
    plot_top_scores([(r[0], r[1], r[2]) for r in rows],
                     title="등록된 커스텀 소리와의 유사도", threshold=0.72)


## 4. 내부적 처리 방법 — YAMNet은 소리를 어떻게 숫자로 바꾸는가

YAMNet의 처리 과정은 크게 4단계입니다.

1. **입력**: 16kHz, mono, -1~1 범위 float 파형 (그래서 `serve.py`도 마이크
   원신호를 48k→16kHz로 낮춰서 보냅니다)
2. **STFT → 로그 멜 스펙트로그램**: 25ms 창을 10ms씩 옮겨가며 주파수 분석
   (STFT) 후, 사람 청각과 비슷하게 저주파를 촘촘히·고주파를 성기게 묶는
   64개 멜 필터로 압축 → `log(mel_energy + eps)`. 이게 아래에서 직접 그려볼
   `log_mel` 배열입니다.
3. **패치화 + CNN**: 멜 스펙트로그램을 0.96초(=96프레임) 단위로 잘라
   (0.48초씩 겹치며 이동) MobileNetV1 계열의 경량 CNN에 통과시켜 **1024차원
   임베딩**을 뽑습니다.
4. **분류 head**: 임베딩에 선형층 + 시그모이드를 적용해 **521개 AudioSet
   클래스별 점수**를 출력합니다.

즉, 우리가 2~3번 섹션에서 본 `scores`와 `embeddings`는 이 파이프라인의
마지막 두 산출물이고, 아래에서 그릴 `log_mel`은 **2번 단계, CNN에 들어가기
직전의 표현**입니다 — "YAMNet이 실제로 무엇을 보고 판단하는지"에 가장 가까운
그림입니다.


In [ ]:
plot_internal_pipeline(waveform, sr, log_mel, title=f"파형 → log-mel 스펙트로그램 — {SOUND_NAME}")


위 그래프를 파형과 나란히 보면, 소리 에너지가 몰린 시간대(위 그래프의 파형이
출렁이는 구간)가 아래 스펙트로그램에서 밝은 색 띠로 그대로 나타나는 것을
확인할 수 있습니다. 화재경보(fire_t3)라면 3.2kHz 부근 멜 밴드에 얇고 뚜렷한
가로줄이 주기적으로 나타나야 하고, 박수(clap)라면 넓은 주파수 대역에 걸쳐
순간적인 세로줄(임펄스)이 여러 번 나타나야 합니다. **다른 테스트 사운드로
바꿔가며(섹션 1의 `SOUND_NAME` 변경 후 재실행) 이 대응 관계를 직접
확인해보세요.**

```
파형(1D, 16kHz)
   │  STFT(25ms 창/10ms hop) + 64 멜 필터
   ▼
log-mel 스펙트로그램(시간 x 64)
   │  0.96s 패치(0.48s hop)로 자르기 + MobileNetV1
   ▼
임베딩(패치 x 1024)
   │  선형층 + 시그모이드
   ▼
클래스 점수(패치 x 521)  ──평균──▶  최종 판정 (2번 섹션)
```


## 5. 개선을 위해 고민해야 할 점

여기 나열한 항목들은 이 프로젝트의 실제 `classifier.py` 코드에 이미 반영된
튜닝값들이 **왜 그 숫자인지**, 그리고 **아직 완전히 풀리지 않은 트레이드오프**를
정리한 것입니다. DS/AI입문자 트랙에서 정오탐 분석을 시작할 때 이 목록을
출발점으로 쓰면 됩니다.

### 문턱값(threshold) 튜닝
- `MIN_SCORE = 0.10` — 이보다 낮으면 아예 보고하지 않음. 너무 낮추면 애매한
  오탐이 늘고, 너무 높이면 작은 소리·먼 소리를 놓칩니다.
- 커스텀 소리 유사도 `threshold = 0.72` — 등록한 소리마다 다르게 줄 수도
  있습니다. 지금은 모두 0.72 고정인데, 소리마다 최적값이 다를 가능성이 큽니다.
- `MATCH_MIN_RMS`, `CAPTURE_MIN_RMS` — "조용한 구간은 비교 자체를 생략"하는
  안전장치. 이 기준이 너무 낮으면 침묵이 지문과 오탐되고, 너무 높으면 멀리서
  나는 진짜 소리(부엌 밥솥 등)를 걸러버립니다. `classifier.py` 주석에
  실측 조정 이력이 있습니다.

### 데이터셋/도메인 불일치
- YAMNet은 **AudioSet**(주로 서구권 유튜브 영상)으로 학습되어, 한국 가정에서
  나는 밥솥/세탁기 완료음 같은 소리는 521개 클래스에 없습니다 → few-shot
  임베딩 등록으로 우회 중이지만, **재학습 없이 임베딩 유사도만으로 얼마나
  정확히 구분되는지**는 등록 소리가 늘어날수록 검증이 필요합니다.
- 등록한 소리끼리 서로 헷갈리기 시작하면(예: 여러 가전 완료음이 비슷비슷)
  어떻게 할지 — 문턱 상향? 소리별 개별 문턱? 임베딩 외 추가 특징?

### 실시간성 / 지연
- 노트북 CPU 추론이 0.7초를 넘으면 `classifier.py`가 경고를 출력합니다 —
  실시간 스트리밍(0.5초마다 판정)이 밀리기 시작한다는 뜻. 팀원 노트북마다
  속도가 다를 수 있으니, 본인 환경에서 추론 1회에 얼마나 걸리는지 재보고
  기록해두면 좋습니다 (`%timeit classify_clip(waveform)`로 확인 가능).
- 장기적으로 ESP32 자체에서 경량 모델을 돌릴 수 있을지(TinyML)도 로드맵
  후보로 논의된 적이 있습니다.

### 평가 방법
- 지금은 "몇 개 실측 사례가 잘 됐다"는 정성적 검증 수준 — 정오탐을 표로
  쌓아 precision/recall 같은 정량 지표로 바꾸는 작업이 필요합니다.
- 위험군(화재경보/사이렌/비명 등) 오탐은 사용자를 놀라게 하고, 미탐은
  안전 문제로 직결되므로 두 오류의 비용이 다릅니다 — 단순 정확도보다
  **위험군 재현율(recall)** 을 더 중요하게 볼지 논의가 필요합니다.

## 실습 과제

1. **문턱값 감각 잡기**: `MIN_SCORE`를 0.05 / 0.10 / 0.20으로 바꿔가며
   여러 테스트 사운드를 분류해보고, 어떤 값에서 결과가 달라지는지 표로
   정리해보세요.
2. **오탐 사냥**: 청소기, 헤어드라이어, 문 닫는 소리, 키보드 타이핑 등
   생활 소음을 랩탑 마이크로 녹음해서 분류해보고, 어떤 AudioSet 라벨로
   오분류되는지 기록해보세요. 흥미로운 사례는 GitHub Projects 이슈로
   등록합니다.
3. **커스텀 소리 등록 시뮬레이션**: 본인 목소리로 "도와주세요" 같은 문구를
   3번 녹음해 임베딩을 평균 내고, 서로 다른 발화끼리 코사인 유사도가
   얼마나 되는지 확인해보세요 (섹션 3의 커스텀 소리 비교 코드를 참고).
4. **시간별 패턴 해석**: `fire_t3`와 `horn` 합성음의 시간별 점수 그래프를
   비교해보고, 왜 모양이 다른지(주기적 vs 지속적) 설명해보세요.

## 참고 자료
- [YAMNet TensorFlow Hub 페이지](https://tfhub.dev/google/yamnet/1)
- [AudioSet 온톨로지](https://research.google.com/audioset/ontology/index.html)
- `docs/factsheet.md` — 이 프로젝트의 전체 원리·개발 과정
- `dashboard/classifier.py` — 이 노트북이 재현한 실제 프로덕션 코드
